# Advanced Python callable objects — 18 problems with complete solutions

**Topic:** Python's callable protocol (`__call__`), `functools.partial`, stateful factories for `defaultdict`, caching, callable decorators, performance profiling, descriptors, async callables, and thread-safe coordination.

**Based on the supplied lesson:** the starting concepts are making instances callable, recreating a simplified `partial`, counting `defaultdict` factory invocations, and implementing class-based profiling decorators. The advanced problems below are newly developed extensions rather than claims that they appeared in the source.

**Audience:** intermediate-to-advanced Python learners. **Runtime:** Python 3.10+ and Jupyter/IPython with standard-library modules only. Run **Kernel → Restart & Run All** to reproduce the checks. Problems are ordered by dependency; solve them yourself before reading each solution.

### Learning outcomes

1. Distinguish callable *instances* from callable *classes*, functions, coroutine-producing calls, and descriptor binding.
2. Design callable objects with correct argument forwarding, encapsulated state, input validation, and explicit invariants.
3. Understand what `defaultdict` can and cannot do, including the difference between a factory call and a key-aware cache miss.
4. Build bounded caches, safe instrumentation, class decorators, and method-aware descriptors.
5. Reason about exceptions, metadata, hashing, concurrency, and resource usage using executable assertions.

### Notebook conventions

Every problem supplies a specification, implementation, examples, assertions, complexity or design notes, and an optional extension. Assertions do not depend on timing thresholds or external network services. Some comparisons are approximate only where stated. Unless specified otherwise, these components are intended for illustrative use, not a production caching/observability package.

**Important corrections/nuances:** a `property` descriptor itself is not generally callable (its getter is a function and is callable). A callable class is not the same thing as a callable instance. `defaultdict`'s zero-argument factory does not receive the missing key. A basic class-based decorator does not automatically bind as an instance method. The exercises test these distinctions explicitly.

## Setup — imports, reusable assertion helper, and reproducibility

In [1]:
import asyncio
import gc
import inspect
import math
import threading
import time
import weakref
from collections import OrderedDict, defaultdict
from concurrent.futures import Future, ThreadPoolExecutor
from dataclasses import dataclass, field
from functools import partial, update_wrapper, wraps
from types import MethodType
from typing import Any, Callable


def assert_raises(exception_type, action, *, contains=None):
    """Assert an action raises a particular exception; return the exception."""
    try:
        action()
    except exception_type as exc:
        if contains is not None:
            assert contains in str(exc), (contains, str(exc))
        return exc
    except BaseException as exc:
        raise AssertionError(
            f"Expected {exception_type.__name__}; got {type(exc).__name__}"
        ) from exc
    raise AssertionError(f"Expected {exception_type.__name__}, but nothing was raised")

print('Setup complete: standard library only.')

Setup complete: standard library only.


---
## Problem 01 — Special-method lookup and the real meaning of callable

**Your task**

Create `Greeter` so that `g('Ada')` works and `callable(g)` is true. Determine what happens if you assign a *different* `__call__` onto only `g`, and contrast it with a class that has no `__call__`. Verify the status of a `property` object. Do not confuse "callable" with "a successful call for every argument list."

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

Python's invocation machinery resolves special methods such as `__call__` on the **type**, not by ordinary lookup of a same-named attribute on an individual instance. The instance attribute can be accessed as `g.__call__`, but it is not the method used by `g(...)`. Classes are usually callable through their metaclass, independently of their instances. `callable(x)` checks whether calling is supported; it cannot promise that arbitrary arguments or the implementation will succeed.

In [2]:
class Greeter:
    def __init__(self, prefix: str = 'Hello') -> None:
        self.prefix = prefix

    def __call__(self, name: str) -> str:
        return f'{self.prefix}, {name}!'


class PlainRecord:
    def __init__(self, value):
        self.value = value


greeter = Greeter()
greeter.__call__ = lambda name: f'OVERRIDE: {name}'  # ordinary instance attribute

### Executable examples and correctness checks

In [3]:
assert callable(Greeter) and callable(greeter)
assert greeter('Ada') == 'Hello, Ada!'
assert greeter.__call__('Ada') == 'OVERRIDE: Ada'
assert callable(PlainRecord) and not callable(PlainRecord(3))
assert callable(Greeter.__call__)
assert not callable(property(lambda self: self.prefix))
assert_raises(TypeError, lambda: greeter())  # missing required argument
print('P01 PASS — special lookup, constructor, and property distinctions')

P01 PASS — special lookup, constructor, and property distinctions


**Review / further challenge.** Explain why changing `Greeter.__call__` at the class level changes invocation of existing instances. Consider why `callable()` should not be used as an argument-signature validator.

---
## Problem 02 — Immutable callable affine transforms and algebraic composition

**Your task**

Implement an immutable callable object `Affine(a, b)` representing `f(x) = a*x + b`. Provide `f(x)`, a `compose(inner)` method returning `f(inner(x))`, and an identity constructor. Require finite real numeric coefficients, reject booleans, and return a new transform on composition. Test algebraic equivalence over several inputs and ensure the inputs remain unchanged.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

A frozen dataclass exposes state clearly and prevents accidental coefficient mutation. Composition is `(a1, b1) ∘ (a2, b2) = (a1*a2, a1*b2+b1)`. Validate the constructed coefficients; floating-point arithmetic uses a tolerance for noninteger results. This is a useful pattern for a reusable, configurable callable with nontrivial behavior.

In [4]:
@dataclass(frozen=True)
class Affine:
    a: float
    b: float

    def __post_init__(self):
        for name in ('a', 'b'):
            value = getattr(self, name)
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                raise TypeError(f'{name} must be a real number, not bool')
            if not math.isfinite(value):
                raise ValueError(f'{name} must be finite')

    def __call__(self, x: float) -> float:
        return self.a * x + self.b

    def compose(self, inner: 'Affine') -> 'Affine':
        if not isinstance(inner, Affine):
            raise TypeError('inner must be an Affine transform')
        return Affine(self.a * inner.a, self.a * inner.b + self.b)

    @classmethod
    def identity(cls) -> 'Affine':
        return cls(1, 0)

### Executable examples and correctness checks

In [5]:
outer, inner = Affine(2, 3), Affine(4, -1)
combined = outer.compose(inner)
assert combined == Affine(8, 1)
for x in (-7, 0, 1.25, 11):
    assert math.isclose(combined(x), outer(inner(x)))
    assert outer.compose(Affine.identity())(x) == outer(x)
    assert Affine.identity().compose(inner)(x) == inner(x)
assert (outer.a, outer.b) == (2, 3)
assert_raises(TypeError, lambda: Affine(True, 2))
assert_raises(ValueError, lambda: Affine(float('inf'), 2))
assert_raises(TypeError, lambda: outer.compose(lambda x: x))
print('P02 PASS — immutable callables and composition algebra')

P02 PASS — immutable callables and composition algebra


**Review / further challenge.** Analyze operation count and memory: `__call__` is O(1); `compose` is O(1). Why might a frozen dataclass be preferable to an object whose coefficients mutate after registering it in a processing pipeline?

---
## Problem 03 — Re-create positional AND keyword partial application

**Your task**

Extend the lesson's positional-only `Partial` approximation. Implement `PartialPlus(func, /, *preset_args, **preset_kwargs)` that validates the wrapped object, forwards new arguments, and lets *call-time keyword arguments override preset keywords*. Include introspection attributes `func`, `args`, and `keywords`. Compare against `functools.partial` for valid test cases. State the deliberate limitations.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

New positional arguments come after stored positional arguments. Copy preset keyword arguments before overriding them, so one invocation does not mutate the next. The object stores references to positional argument objects, like ordinary Python argument passing. This implementation mimics common call-forwarding behavior but **does not** reproduce every feature of `functools.partial` (e.g., its complete introspection, serialization, descriptor and version-specific features).

In [6]:
class PartialPlus:
    def __init__(self, func, /, *preset_args, **preset_kwargs):
        if not callable(func):
            raise TypeError('func must be callable')
        self.func = func
        self.args = preset_args
        self.keywords = dict(preset_kwargs)

    def __call__(self, /, *args, **kwargs):
        merged_kwargs = {**self.keywords, **kwargs}
        return self.func(*self.args, *args, **merged_kwargs)


def build_label(a, b, *, prefix='ID', suffix='!'):
    return f'{prefix}:{a}-{b}{suffix}'

custom_partial = PartialPlus(build_label, 4, prefix='OLD')
standard_partial = partial(build_label, 4, prefix='OLD')

### Executable examples and correctness checks

In [7]:
for kwargs in ({}, {'prefix': 'NEW'}, {'suffix': '?'}, {'prefix': 'P', 'suffix': '#'}):
    assert custom_partial(9, **kwargs) == standard_partial(9, **kwargs)
assert custom_partial(9, prefix='NEW') == 'NEW:4-9!'
assert custom_partial(9) == 'OLD:4-9!'  # preset unchanged
assert custom_partial.args == (4,)
assert custom_partial.keywords == {'prefix': 'OLD'}
assert_raises(TypeError, lambda: PartialPlus(123))
assert_raises(TypeError, lambda: custom_partial(9, a=10))  # duplicate a
print('P03 PASS — keyword overrides and argument forwarding')

P03 PASS — keyword overrides and argument forwarding


**Review / further challenge.** Compare `inspect.signature(custom_partial)` with `inspect.signature(standard_partial)` in your environment. For a production library, prefer standard `functools.partial` unless you need substantially different semantics.

---
## Problem 04 — Independent stateful defaultdict factories

**Your task**

Construct `MissFactory(default)` with `__call__(self)` and a counter. Use **two separate factories** to create two `defaultdict` instances. Show that repeated access to an existing key is a hit, `get()` does not trigger the factory, and `setdefault()` inserts its own supplied value without calling the factory. Demonstrate independent counters, including when a default is `None`.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

`defaultdict.__missing__` uses a zero-argument `default_factory` only for a missing `__getitem__` lookup. The factory sees no key; its state is local to its object, unlike the lesson's rejected global-counter approach. `get`, membership checks, and `setdefault` have distinct behavior. If you pass **one factory instance** to two dictionaries, the counter is shared; isolate instances when you need independent accounting.

In [8]:
class MissFactory:
    def __init__(self, default):
        self.default = default
        self.calls = 0

    def __call__(self):
        self.calls += 1
        return self.default


factory_a = MissFactory(None)
factory_b = MissFactory(0)
cache_a = defaultdict(factory_a)
cache_b = defaultdict(factory_b)

### Executable examples and correctness checks

In [9]:
assert cache_a.get('absent') is None and factory_a.calls == 0
assert 'absent' not in cache_a
assert cache_a['x'] is None and cache_a['x'] is None
assert factory_a.calls == 1 and list(cache_a) == ['x']
assert cache_a.setdefault('y', 88) == 88
assert factory_a.calls == 1
assert cache_b['x'] == 0 and cache_b['y'] == 0
assert factory_b.calls == 2 and factory_a.calls == 1
shared = MissFactory('missing')
left, right = defaultdict(shared), defaultdict(shared)
assert left['l'] == right['r'] == 'missing' and shared.calls == 2
print('P04 PASS — accurate defaultdict factory invocation semantics')

P04 PASS — accurate defaultdict factory invocation semantics


**Review / further challenge.** A factory returning a mutable object should usually make a **new** object per missing key. Try a callable returning `[]` instead of returning one stored shared list, and show whether modifications to `d["a"]` affect `d["b"]`.

---
## Problem 05 — A key-aware cache using dict.__missing__

**Your task**

Build `KeyAwareCache(loader)` so `cache[key]` invokes `loader(key)` only if the key is missing, then stores the result. Keep hit and miss counts. If loading raises an exception, do not store an entry. Explain how this differs from `defaultdict`'s no-argument factory and why `.get()` bypasses your `__missing__` hook.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

A `dict` subclass's `__missing__(key)` receives the missing key when `__getitem__` is used. Implement `__getitem__` to count existing-key hits and delegate misses to the built-in dictionary lookup; the base lookup invokes `__missing__`. Count a failed loading attempt as a miss, but cache only a successful result, including legitimate `None` values. The hook is **not** invoked by `get()`.

In [10]:
class KeyAwareCache(dict):
    def __init__(self, loader):
        if not callable(loader):
            raise TypeError('loader must be callable')
        super().__init__()
        self.loader = loader
        self.hits = 0
        self.misses = 0

    def __getitem__(self, key):
        if key in self:
            self.hits += 1
        return super().__getitem__(key)

    def __missing__(self, key):
        self.misses += 1
        value = self.loader(key)  # an exception leaves the cache unchanged
        self[key] = value
        return value


loaded_keys = []

def uppercase_loader(key):
    loaded_keys.append(key)
    if key == 'bad':
        raise ValueError('load failed')
    return key.upper() if key != 'empty' else None

key_cache = KeyAwareCache(uppercase_loader)

### Executable examples and correctness checks

In [11]:
assert key_cache.get('a') is None and key_cache.misses == 0
assert key_cache['a'] == 'A' and key_cache['a'] == 'A'
assert key_cache['empty'] is None and key_cache['empty'] is None
assert (key_cache.hits, key_cache.misses) == (2, 2)
assert loaded_keys == ['a', 'empty']
assert_raises(ValueError, lambda: key_cache['bad'], contains='failed')
assert 'bad' not in key_cache and key_cache.misses == 3
assert_raises(ValueError, lambda: key_cache['bad'])
assert key_cache.misses == 4 and loaded_keys[-2:] == ['bad', 'bad']
print('P05 PASS — key-aware loading and exception-safe insertion')

P05 PASS — key-aware loading and exception-safe insertion


**Review / further challenge.** What should `cache.setdefault(key, value)` do with hit/miss counters? Our contract counts only square-bracket lookups, not every possible dictionary operation. Write that contract down before attempting to intercept all access paths.

---
## Problem 06 — Bounded LRU memoization with canonical argument binding

**Your task**

Implement a callable `BoundedMemoizer(func, maxsize)` that stores results using an `OrderedDict`, tracks hits and misses, and evicts the least-recently used key when full. Calls equivalent after signature binding and default expansion (such as `f(2)`, `f(x=2)` and `f(2, y=3)` for a default `y=3`) must use the same entry. Cache `None` correctly; do not cache failures. Reject unhashable argument values and nonpositive capacity.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

Use `inspect.signature(...).bind(*args, **kwargs)` plus `apply_defaults()` to normalize valid argument styles. A tuple of bound `(parameter_name, value)` pairs forms the key; this requires all values to be hashable. `OrderedDict.move_to_end()` marks a hit as recently used. Evict with `popitem(last=False)`. This is a *single-threaded educational cache*: it does not automatically handle mutable inputs, custom equivalence, time-to-live or concurrent duplication. For typical applications, prefer `functools.lru_cache` when its key semantics are appropriate.

In [12]:
class BoundedMemoizer:
    def __init__(self, func, maxsize=128):
        if not callable(func):
            raise TypeError('func must be callable')
        if type(maxsize) is not int or maxsize <= 0:
            raise ValueError('maxsize must be a positive integer')
        self.func = func
        self.maxsize = maxsize
        self.signature = inspect.signature(func)
        self.cache = OrderedDict()
        self.hits = 0
        self.misses = 0
        update_wrapper(self, func)

    def __call__(self, *args, **kwargs):
        bound = self.signature.bind(*args, **kwargs)
        bound.apply_defaults()
        key = tuple(bound.arguments.items())
        # Forces clear TypeError on unhashable bound values, even for empty cache.
        hash(key)
        if key in self.cache:
            self.hits += 1
            self.cache.move_to_end(key)
            return self.cache[key]
        self.misses += 1
        result = self.func(*args, **kwargs)
        self.cache[key] = result
        if len(self.cache) > self.maxsize:
            self.cache.popitem(last=False)
        return result

    def cache_clear(self):
        self.cache.clear()
        self.hits = self.misses = 0


compute_log = []

def memo_target(x, y=3):
    compute_log.append((x, y))
    if x == -1:
        raise ArithmeticError('cannot compute')
    return None if x == 0 else x + y

memo = BoundedMemoizer(memo_target, maxsize=2)

### Executable examples and correctness checks

In [13]:
assert memo(2) == memo(x=2) == memo(2, y=3) == 5
assert (memo.hits, memo.misses) == (2, 1)
assert compute_log == [(2, 3)]
assert memo(0) is None and memo(x=0) is None  # None is cached
assert (memo.hits, memo.misses) == (3, 2)
assert memo(2) == 5  # make 2 most recent
assert memo(4) == 7  # evicts 0
assert memo(0) is None  # recompute evicted entry
assert compute_log.count((0, 3)) == 2
size_before_error = len(memo.cache)
assert_raises(ArithmeticError, lambda: memo(-1))
assert len(memo.cache) == size_before_error
assert_raises(TypeError, lambda: memo([2]))
assert_raises(ValueError, lambda: BoundedMemoizer(memo_target, 0))
assert_raises(ValueError, lambda: BoundedMemoizer(memo_target, True))
print('P06 PASS — canonical argument keys, LRU eviction, None, failures')

P06 PASS — canonical argument keys, LRU eviction, None, failures


**Review / further challenge.** Complexity: signature binding costs O(number of arguments); expected dictionary lookup and OrderedDict recency updates are O(1); cache space is O(maxsize × entry size). Inspect `functools.lru_cache` for a mature alternative. Why do mutable input arguments create both hashing and correctness problems?

---
## Problem 07 — Exception-safe profiler with injectable clock

**Your task**

Create a callable profiling wrapper that maintains total calls, successful calls, failures, total elapsed time, and a lazy average. It must preserve the original return value, update duration *even if the wrapped function raises*, expose a zero-call average without division by zero, and allow a fake clock for deterministic tests. Preserve `__name__` and `__wrapped__` where possible.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

A `try`/`except`/`finally` combination accounts for success and failure without swallowing exceptions. Put time accumulation in `finally`. Use an injectable clock instead of `sleep()` or unstable elapsed-time assertions. `functools.update_wrapper` adds useful metadata; it does **not** turn this class into a method descriptor (that is a separate problem). Duration is wall-clock elapsed time, not CPU time, and includes time spent inside the wrapped call.

In [14]:
class SafeProfiler:
    def __init__(self, fn, *, clock=time.perf_counter):
        if not callable(fn) or not callable(clock):
            raise TypeError('fn and clock must be callable')
        self.fn = fn
        self.clock = clock
        self.calls = 0
        self.successes = 0
        self.failures = 0
        self.total_seconds = 0.0
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        start = self.clock()
        self.calls += 1
        try:
            result = self.fn(*args, **kwargs)
        except BaseException:
            self.failures += 1
            raise
        else:
            self.successes += 1
            return result
        finally:
            self.total_seconds += self.clock() - start

    @property
    def average_seconds(self):
        return self.total_seconds / self.calls if self.calls else 0.0


def checked_divide(a, b):
    """Divide a by b for an instrumented demo."""
    return a / b

clock_ticks = iter([1.0, 1.5, 2.0, 2.25])
profiled_divide = SafeProfiler(checked_divide, clock=lambda: next(clock_ticks))

### Executable examples and correctness checks

In [15]:
assert profiled_divide.average_seconds == 0.0
assert profiled_divide(8, 2) == 4
assert_raises(ZeroDivisionError, lambda: profiled_divide(8, 0))
assert (profiled_divide.calls, profiled_divide.successes, profiled_divide.failures) == (2, 1, 1)
assert math.isclose(profiled_divide.total_seconds, 0.75)
assert math.isclose(profiled_divide.average_seconds, 0.375)
assert profiled_divide.__name__ == 'checked_divide'
assert profiled_divide.__wrapped__ is checked_divide
print('P07 PASS — measured successes and failures with deterministic time')

P07 PASS — measured successes and failures with deterministic time


**Review / further challenge.** If the timer itself raises inside `finally`, it can mask the original error. Discuss whether an observability wrapper should catch timer failures or require a trustworthy clock. For an async function, timing the creation of a coroutine is NOT timing its execution; see Problem 13.

---
## Problem 08 — Make a class-based decorator work on instance methods

**Your task**

The lesson's simplest `Profiler` object decorates module-level functions, but `@Profiler` applied directly to an ordinary instance method needs method binding. Implement `DescriptorProfiler` with `__get__`, `__call__`, and call counting. Check access through the class, through two instances, and by calling an ordinary decorated function. Explicitly define whether its counter is shared across instances.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

When a decorated attribute is an object instead of a function, automatic function binding is lost unless the object supports the descriptor protocol. `__get__(self, instance, owner)` should return the decorator itself for class access and `MethodType(self, instance)` for an instance. This inserts the instance as the first argument to `__call__`. This particular profiler stores **one counter on the descriptor**, so usage from multiple instances is aggregated.

In [16]:
class DescriptorProfiler:
    def __init__(self, fn):
        self.fn = fn
        self.calls = 0
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.fn(*args, **kwargs)

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return MethodType(self, instance)


class Calculator:
    def __init__(self, base):
        self.base = base

    @DescriptorProfiler
    def add(self, n):
        return self.base + n


@DescriptorProfiler
def standalone_double(x):
    return 2 * x

### Executable examples and correctness checks

In [17]:
first, second = Calculator(10), Calculator(100)
assert first.add(3) == 13
assert second.add(3) == 103
assert first.add(n=5) == 15
assert Calculator.add.calls == 3  # deliberately aggregated
assert Calculator.add(first, 7) == 17
assert Calculator.add.calls == 4
assert standalone_double(9) == 18 and standalone_double.calls == 1
assert first.add.__self__ is first
print('P08 PASS — descriptor binding and shared instrumentation')

P08 PASS — descriptor binding and shared instrumentation


**Review / further challenge.** Explain why `@staticmethod` or `@classmethod` combined with a custom descriptor depends on decorator order. Test orders separately before relying on them; this demonstration intentionally targets ordinary instance methods.

---
## Problem 09 — Per-instance method metrics without retaining every instance

**Your task**

Build a method decorator descriptor that tracks call counts **individually** for each owner instance rather than aggregating counts. Store state using `weakref.WeakKeyDictionary`, preserve decorated-method metadata, and provide `stats_for(obj)` on the descriptor. Confirm that calls on one object do not change another object's metrics. Document instance requirements.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

The descriptor returns a small closure that captures the accessed instance and updates a corresponding record. A weak-key map does not keep the instance alive merely because statistics exist. Its keys must be weak-referenceable and hashable; many ordinary Python instances qualify, while some slotted or unhashable classes do not. This version is not synchronized for threaded use. It stores simple integer data as values, never the instance, to avoid a strong reference cycle in the metrics map.

In [18]:
@dataclass
class PerInstanceStats:
    calls: int = 0
    failures: int = 0


class PerInstanceProfiler:
    def __init__(self, fn):
        self.fn = fn
        self._stats = weakref.WeakKeyDictionary()
        update_wrapper(self, fn)

    def stats_for(self, instance):
        if instance not in self._stats:
            self._stats[instance] = PerInstanceStats()
        return self._stats[instance]

    def __get__(self, instance, owner=None):
        if instance is None:
            return self

        @wraps(self.fn)
        def bound(*args, **kwargs):
            stats = self.stats_for(instance)
            stats.calls += 1
            try:
                return self.fn(instance, *args, **kwargs)
            except BaseException:
                stats.failures += 1
                raise

        return bound


class BankAccount:
    def __init__(self, balance):
        self.balance = balance

    @PerInstanceProfiler
    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError('insufficient funds')
        self.balance -= amount
        return self.balance

### Executable examples and correctness checks

In [19]:
account_a, account_b = BankAccount(20), BankAccount(30)
assert account_a.withdraw(5) == 15
assert account_b.withdraw(10) == 20
assert_raises(ValueError, lambda: account_a.withdraw(50))
stats_a = BankAccount.withdraw.stats_for(account_a)
stats_b = BankAccount.withdraw.stats_for(account_b)
assert (stats_a.calls, stats_a.failures) == (2, 1)
assert (stats_b.calls, stats_b.failures) == (1, 0)
assert account_a.withdraw.__name__ == 'withdraw'
# Do not keep any bound method referencing an ephemeral account.
def weak_key_cleanup_demo():
    ephemeral = BankAccount(1)
    BankAccount.withdraw.stats_for(ephemeral)
    return weakref.ref(ephemeral)

ephemeral_ref = weak_key_cleanup_demo()
gc.collect()
assert ephemeral_ref() is None
print('P09 PASS — isolated per-instance metrics and weak ownership')

P09 PASS — isolated per-instance metrics and weak ownership


**Review / further challenge.** If your objects implement value-based `__eq__`/`__hash__`, a `WeakKeyDictionary` may not represent identity-based ownership as you expect. Investigate an instance-attached stats object or another identity-aware design when such classes must be supported.

---
## Problem 10 — Parameterized retry decorator as a callable configuration object

**Your task**

Create a callable `Retry(times=3, exceptions=(ValueError,))` used as `@Retry(...)` to retry a decorated function up to the specified total number of attempts. Validate configuration, preserve metadata using `wraps`, retry only selected exceptions, return immediately after success, and re-raise the last selected exception after exhaustion. No sleeping, jitter, or network calls are needed.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

There are two different calls: `Retry(...)` constructs the configuration object, and `Retry(...)(fn)` uses that object's `__call__` to produce a wrapper. The produced wrapper must itself accept `*args, **kwargs`. Looping a bounded number of attempts and using bare `raise` inside `except` preserves the last exception traceback. Never catch every exception indiscriminately: the caller needs to know which failures are retryable. This is a teaching example, not a general retry policy for non-idempotent operations.

In [20]:
class Retry:
    def __init__(self, times=3, exceptions=(ValueError,)):
        if type(times) is not int or times < 1:
            raise ValueError('times must be a positive integer')
        if (not isinstance(exceptions, tuple) or not exceptions
                or not all(isinstance(exc, type) and issubclass(exc, Exception)
                           for exc in exceptions)):
            raise TypeError('exceptions must be a nonempty tuple of Exception classes')
        self.times = times
        self.exceptions = exceptions

    def __call__(self, fn):
        if not callable(fn):
            raise TypeError('decorated object must be callable')

        @wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, self.times + 1):
                try:
                    return fn(*args, **kwargs)
                except self.exceptions:
                    if attempt == self.times:
                        raise
            raise AssertionError('unreachable')

        return wrapper


attempts = []

@Retry(times=4, exceptions=(ValueError,))
def eventually_succeeds(value):
    attempts.append(value)
    if len(attempts) < 3:
        raise ValueError('temporary')
    return value * 2

### Executable examples and correctness checks

In [21]:
assert eventually_succeeds(7) == 14
assert attempts == [7, 7, 7]
assert eventually_succeeds.__name__ == 'eventually_succeeds'
failures = []
@Retry(times=2, exceptions=(ValueError,))
def always_fails():
    failures.append('attempt')
    raise ValueError('still failing')
assert_raises(ValueError, always_fails, contains='still failing')
assert len(failures) == 2
non_retry = []
@Retry(times=5, exceptions=(ValueError,))
def different_error():
    non_retry.append(1)
    raise TypeError('not retryable')
assert_raises(TypeError, different_error)
assert len(non_retry) == 1
assert_raises(ValueError, lambda: Retry(times=0))
assert_raises(TypeError, lambda: Retry(exceptions=(ValueError, 42)))
print('P10 PASS — parameterized decorator and strict error scope')

P10 PASS — parameterized decorator and strict error scope


**Review / further challenge.** Explain why retrying a payment or other side-effecting operation is unsafe without an idempotency guarantee. Add optional backoff by injecting a sleeper into the decorator; avoid real sleeps in unit tests.

---
## Problem 11 — Composable callables with short-circuiting and operator overloading

**Your task**

Implement `Pipeline(*steps)` so calling it passes one value through each callable in order. Support composition with `pipeline | function` and `pipeline | another_pipeline`; return a new pipeline, leaving the original unchanged. An empty pipeline must act as the identity. Invalid steps should fail fast; if one step raises, later steps must not run.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

The pipeline is a callable strategy object that stores an immutable tuple of callable stages. A small `__or__` method makes composition readable without changing prior objects. Argument forwarding is deliberately **unary**: each step accepts one value, so the contract is not confused with a general multiparameter partial. Exceptions propagate naturally and stop evaluation at that step.

In [22]:
class Pipeline:
    def __init__(self, *steps):
        if any(not callable(step) for step in steps):
            raise TypeError('every pipeline step must be callable')
        self.steps = tuple(steps)

    def __call__(self, value):
        for step in self.steps:
            value = step(value)
        return value

    def __or__(self, other):
        if isinstance(other, Pipeline):
            return Pipeline(*self.steps, *other.steps)
        if callable(other):
            return Pipeline(*self.steps, other)
        return NotImplemented


strip = Pipeline(str.strip)
normalize = strip | str.casefold | (lambda text: text.replace(' ', '_'))

### Executable examples and correctness checks

In [23]:
assert Pipeline()(42) == 42
assert normalize('  Hello WORLD  ') == 'hello_world'
assert strip('  A B  ') == 'A B'
assert len(strip.steps) == 1 and len(normalize.steps) == 3
assert (Pipeline(lambda n: n + 1) | Pipeline(lambda n: n * 10))(2) == 30
assert_raises(TypeError, lambda: Pipeline(lambda x: x, 123))
assert_raises(TypeError, lambda: normalize | 123)
events = []
def explode(value):
    events.append('explode')
    raise RuntimeError('stop')
def too_late(value):
    events.append('too_late')
    return value
assert_raises(RuntimeError, lambda: Pipeline(explode, too_late)('input'))
assert events == ['explode']
print('P11 PASS — immutable composition and exception short-circuit')

P11 PASS — immutable composition and exception short-circuit


**Review / further challenge.** Find a real preprocessing task where validation should come before transformation. How would you support steps that need more than one input without turning every stage into a brittle special case?

---
## Problem 12 — Callable strategy registry with validation and dispatch

**Your task**

Implement a `StrategyRouter` with `register(name, strategy)` and `router(name, *args, **kwargs)`. Validate the name and callable, refuse duplicate registrations unless `replace=True`, raise an informative `KeyError` for unknown names, and dispatch equally to functions, callable objects, and bound methods. Show a partial function used as a registered strategy.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

Callables unify different implementation styles through one interface. A registry should not assume `type(strategy)` is a function; it should check `callable(strategy)` and let the actual call validate argument signatures. Registration is intentionally separate from invocation, so an unknown name is not accidentally treated as a callable value.

In [24]:
class StrategyRouter:
    def __init__(self):
        self._strategies = {}

    def register(self, name, strategy, *, replace=False):
        if not isinstance(name, str) or not name.strip():
            raise ValueError('strategy name must be a nonempty string')
        if not callable(strategy):
            raise TypeError('strategy must be callable')
        if name in self._strategies and not replace:
            raise KeyError(f'already registered: {name}')
        self._strategies[name] = strategy

    def __call__(self, name, *args, **kwargs):
        try:
            strategy = self._strategies[name]
        except KeyError:
            raise KeyError(f'unknown strategy: {name!r}') from None
        return strategy(*args, **kwargs)


router = StrategyRouter()
router.register('double', lambda value: value * 2)
router.register('linear', Affine(3, 1))
router.register('label', partial(build_label, 5, prefix='USER'))

### Executable examples and correctness checks

In [25]:
assert router('double', 11) == 22
assert router('linear', 4) == 13
assert router('label', 7) == 'USER:5-7!'
assert_raises(KeyError, lambda: router.register('double', str))
router.register('double', lambda value: 4 * value, replace=True)
assert router('double', 3) == 12
assert_raises(KeyError, lambda: router('missing', 1), contains='unknown strategy')
assert_raises(TypeError, lambda: router.register('broken', object()))
assert_raises(ValueError, lambda: router.register('  ', str))
print('P12 PASS — polymorphic dispatch and explicit registration policy')

P12 PASS — polymorphic dispatch and explicit registration policy


**Review / further challenge.** Consider adding `unregister` and `available_names` without exposing the internal dictionary for arbitrary mutation. Do you want names to be case-sensitive? Make that a documented policy rather than silently normalizing some but not all paths.

---
## Problem 13 — Asynchronous callable objects: creation versus execution

**Your task**

Implement `AsyncTransformer` whose `async def __call__` validates an integer and returns a transformed value after an asynchronous scheduling point. Check `callable(transformer)`, demonstrate that invoking it produces an awaitable, and use **top-level `await` in a Jupyter cell** to obtain a result. Demonstrate an exception occurring when awaited. Explain why a synchronous profiler around an async function measures coroutine creation rather than awaited work.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

`async def __call__` makes the instance callable, but `obj(x)` returns a coroutine (it does not itself run the body to completion). An `await` is required. Jupyter supports top-level `await`, whereas `asyncio.run()` is often inappropriate inside an already running notebook event loop. Avoid creating unawaited coroutines in tests; create one and await that exact object. A true async profiler should use `async def __call__` and place `await wrapped(...)` inside its timed `try/finally`.

In [26]:
class AsyncTransformer:
    def __init__(self, scale):
        self.scale = scale
        self.invocations = 0

    async def __call__(self, x):
        self.invocations += 1
        if type(x) is not int:
            raise TypeError('x must be an integer')
        await asyncio.sleep(0)  # yield control, not a wall-clock delay
        return self.scale * x


class AsyncProfiler:
    def __init__(self, fn, *, clock=time.perf_counter):
        self.fn = fn
        self.clock = clock
        self.calls = 0
        self.total_seconds = 0.0
        update_wrapper(self, fn)

    async def __call__(self, *args, **kwargs):
        start = self.clock()
        self.calls += 1
        try:
            return await self.fn(*args, **kwargs)
        finally:
            self.total_seconds += self.clock() - start


async_transform = AsyncTransformer(scale=6)

### Executable examples and correctness checks

In [27]:
assert callable(async_transform)
coroutine = async_transform(7)
assert inspect.isawaitable(coroutine)
assert async_transform.invocations == 0  # body has not yet run
assert await coroutine == 42
assert async_transform.invocations == 1
try:
    await async_transform('bad')
except TypeError as error:
    assert 'integer' in str(error)
else:
    raise AssertionError('expected TypeError during await')
assert async_transform.invocations == 2
async_profile = AsyncProfiler(async_transform)
assert await async_profile(3) == 18
assert async_profile.calls == 1 and async_profile.total_seconds >= 0
print('P13 PASS — async invocation and timing of awaited execution')

P13 PASS — async invocation and timing of awaited execution


**Review / further challenge.** AsyncProfiler expects the wrapped callable to return an awaitable. Consider recording successes/failures as in Problem 7. A single timing value is not an event-loop utilization metric; distinguish latency from CPU utilization.

---
## Problem 14 — Thread-safe call counter without serializing expensive work

**Your task**

Build `ThreadSafeCounter(fn)` that counts total calls and failures correctly when invoked by multiple threads. Use a lock only when modifying shared counters; do **not** hold the lock while executing `fn`, because this would serialize useful work. Test with `ThreadPoolExecutor` and deterministic returned values, including a separately tested error path.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

Threads can interleave reads and writes to shared state; do not treat bytecode-level behavior or the GIL as a concurrency guarantee for general Python implementations. Lock only critical sections. Increase total calls before invocation, and increase failures inside the exception path. Errors still propagate. This counter guarantees its own metric consistency; it cannot make the *wrapped function's* state thread-safe.

In [28]:
class ThreadSafeCounter:
    def __init__(self, fn):
        if not callable(fn):
            raise TypeError('fn must be callable')
        self.fn = fn
        self._lock = threading.Lock()
        self.calls = 0
        self.failures = 0
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        with self._lock:
            self.calls += 1
        try:
            return self.fn(*args, **kwargs)
        except BaseException:
            with self._lock:
                self.failures += 1
            raise


@ThreadSafeCounter
def square_or_fail(number):
    if number < 0:
        raise ValueError('negative input')
    return number * number

### Executable examples and correctness checks

In [29]:
inputs = list(range(100))
with ThreadPoolExecutor(max_workers=8) as executor:
    results = list(executor.map(square_or_fail, inputs))
assert results == [n * n for n in inputs]
assert square_or_fail.calls == 100 and square_or_fail.failures == 0
assert_raises(ValueError, lambda: square_or_fail(-1))
assert square_or_fail.calls == 101 and square_or_fail.failures == 1
print('P14 PASS — thread-safe counters, no lock around wrapped work')

P14 PASS — thread-safe counters, no lock around wrapped work


**Review / further challenge.** If you add a duration average, protect the total-duration update and the paired metric snapshot. To test concurrency bugs, use a barrier or controlled scheduling and repeat the test under multiple interpreter implementations where available.

---
## Problem 15 — Signature-aware callable adapter with explicit parameter policy

**Your task**

Create `KeywordAdapter(fn, aliases)` that translates specified old keyword names to new ones before invoking `fn`. Reject ambiguous calls supplying both an old name and its new name, reject alias maps with collisions, and use `inspect.signature.bind` to surface incorrect signatures **before** calling the function. Keep positional arguments intact and support callable instances.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

An adapter can keep a public API stable during a parameter rename. Check name-to-name aliases at construction. At call time, copy the caller's kwargs, reject duplicate old/new inputs, translate, validate the final call with the wrapped callable's signature, and only then call it. `inspect.signature` may not support every arbitrary built-in or extension callable; that limitation is explicit in this exercise.

In [30]:
class KeywordAdapter:
    def __init__(self, fn, aliases):
        if not callable(fn):
            raise TypeError('fn must be callable')
        if not isinstance(aliases, dict):
            raise TypeError('aliases must be a dictionary')
        if any(not isinstance(old, str) or not isinstance(new, str) or not old or not new
               for old, new in aliases.items()):
            raise ValueError('alias names must be nonempty strings')
        if any(old == new for old, new in aliases.items()):
            raise ValueError('an alias must actually rename a keyword')
        if len(set(aliases.values())) != len(aliases):
            raise ValueError('two old names cannot map to the same new name')
        if set(aliases).intersection(aliases.values()):
            raise ValueError('alias chains are not supported')
        self.fn = fn
        self.aliases = dict(aliases)
        self.signature = inspect.signature(fn)
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        translated = dict(kwargs)
        for old, new in self.aliases.items():
            if old in translated:
                if new in translated:
                    raise TypeError(f'provide only one of {old!r} and {new!r}')
                translated[new] = translated.pop(old)
        self.signature.bind(*args, **translated)
        return self.fn(*args, **translated)


def combine_terms(*, term, multiplier=1):
    return term * multiplier

adapter = KeywordAdapter(combine_terms, {'word': 'term', 'scale': 'multiplier'})

### Executable examples and correctness checks

In [31]:
assert adapter(word='ha', scale=3) == 'hahaha'
assert adapter(term='ho', multiplier=2) == 'hoho'
assert_raises(TypeError, lambda: adapter(word='a', term='b'), contains='only one')
assert_raises(TypeError, lambda: adapter(scale=4))  # missing required term
assert_raises(ValueError, lambda: KeywordAdapter(combine_terms, {'a': 'x', 'b': 'x'}))
assert_raises(ValueError, lambda: KeywordAdapter(combine_terms, {'a': 'b', 'b': 'c'}))
assert_raises(ValueError, lambda: KeywordAdapter(combine_terms, {'term': 'term'}))
print('P15 PASS — safe keyword translation and signature validation')

P15 PASS — safe keyword translation and signature validation


**Review / further challenge.** A positional argument can still conflict with a translated keyword; `Signature.bind` detects this. How would you emit a deprecation warning exactly once for old names without sharing global variables?

---
## Problem 16 — Single-flight callable cache for concurrent identical work

**Your task**

Build `SingleFlight(fn)` so concurrent callers requesting the **same hashable positional argument tuple** share one calculation while different keys can compute concurrently. Store a `Future` per in-flight or completed key, increment a computation counter once per newly created key, propagate exceptions to all waiters, and remove failed entries so a later call can retry. State limitations clearly.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

Use a lock only to choose which caller owns a key. The owner creates a `concurrent.futures.Future`, releases the lock, evaluates the wrapped function, then sets its result or exception. Followers wait on `future.result()` outside the lock. On failure, remove the key only if it still refers to the owner's future, then publish the exception to existing followers. This is a **simplified** single-flight cache: positional hashable arguments only; recursive same-key calls can deadlock; retained successful results are unbounded; no cancellation or eviction policy.

In [32]:
class SingleFlight:
    def __init__(self, fn):
        if not callable(fn):
            raise TypeError('fn must be callable')
        self.fn = fn
        self._lock = threading.Lock()
        self._futures = {}
        self.computations = 0
        update_wrapper(self, fn)

    def __call__(self, *args):
        key = args
        hash(key)  # reject unhashable inputs before taking the lock
        with self._lock:
            future = self._futures.get(key)
            leader = future is None
            if leader:
                future = Future()
                self._futures[key] = future
                self.computations += 1
        if leader:
            try:
                result = self.fn(*args)
            except BaseException as exc:
                with self._lock:
                    if self._futures.get(key) is future:
                        del self._futures[key]
                future.set_exception(exc)
                raise
            else:
                future.set_result(result)
                return result
        return future.result()


work_log = []
work_log_lock = threading.Lock()
release_work = threading.Event()

def expensive_double(n):
    with work_log_lock:
        work_log.append(n)
    if n == 9:
        release_work.wait(timeout=5)
    return n * 2

single_flight = SingleFlight(expensive_double)

### Executable examples and correctness checks

In [33]:
# The lead worker blocks inside fn until all tasks have been submitted.
with ThreadPoolExecutor(max_workers=8) as executor:
    pending = [executor.submit(single_flight, 9) for _ in range(8)]
    release_work.set()
    values = [future.result(timeout=5) for future in pending]
assert values == [18] * 8
assert single_flight.computations == 1 and work_log == [9]
assert single_flight(9) == 18 and single_flight.computations == 1
assert single_flight(2) == 4 and single_flight.computations == 2
assert_raises(TypeError, lambda: single_flight([1, 2]))
error_attempts = []
def flaky(number):
    error_attempts.append(number)
    if len(error_attempts) == 1:
        raise ValueError('transient')
    return number
flaky_flight = SingleFlight(flaky)
assert_raises(ValueError, lambda: flaky_flight(5))
assert flaky_flight(5) == 5
assert flaky_flight.computations == 2
print('P16 PASS — shared computations, retry after failure, per-key futures')

P16 PASS — shared computations, retry after failure, per-key futures


**Review / further challenge.** Why is it important not to wait on `future.result()` while holding the mutex? What happens if `fn(key)` recursively calls `single_flight(key)`? Design a policy for reentrancy before extending this to arbitrary recursive workloads.

---
## Problem 17 — A class-based decorator factory preserving method semantics

**Your task**

Create `Counted(max_calls)` used as `@Counted(max_calls=2)` on an ordinary instance method. It must count successful and failed *attempts* per decorated method, reject calls after the limit with a custom exception, and preserve automatic method binding and metadata. Demonstrate that the limit is **shared across all instances** for that decorated method. Avoid confusing `@Counted(...)` with `@Counted`.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

A callable configuration object may return a **function wrapper** from its `__call__(fn)` method. Ordinary function wrappers already implement binding as descriptors when placed on classes, so there is no need to write `__get__` in this design. The per-decorated-function state lives in the closure, whereas the configuration object only holds `max_calls`. This shows an alternative to Problem 8, which returns an instance of the decorator class itself.

In [34]:
class CallLimitReached(RuntimeError):
    pass


class Counted:
    def __init__(self, max_calls):
        if type(max_calls) is not int or max_calls < 1:
            raise ValueError('max_calls must be positive')
        self.max_calls = max_calls

    def __call__(self, fn):
        calls = 0

        @wraps(fn)
        def wrapper(*args, **kwargs):
            nonlocal calls
            if calls >= self.max_calls:
                raise CallLimitReached('call limit exhausted')
            calls += 1
            return fn(*args, **kwargs)

        def calls_so_far():
            return calls

        wrapper.calls_so_far = calls_so_far
        return wrapper


class LimitedWorker:
    @Counted(max_calls=2)
    def work(self, amount):
        """Compute a scaled amount."""
        return 10 * amount

### Executable examples and correctness checks

In [35]:
worker_a, worker_b = LimitedWorker(), LimitedWorker()
assert worker_a.work(2) == 20
assert worker_b.work(3) == 30
assert LimitedWorker.work.calls_so_far() == 2
assert_raises(CallLimitReached, lambda: worker_a.work(4))
assert LimitedWorker.work.calls_so_far() == 2  # rejected calls not counted
assert worker_a.work.__name__ == 'work'
assert worker_a.work.__doc__ == 'Compute a scaled amount.'
assert_raises(ValueError, lambda: Counted(0))
print('P17 PASS — class decorator factory returning a bound function')

P17 PASS — class decorator factory returning a bound function


**Review / further challenge.** This wrapper is deliberately not thread-safe: two threads can pass the limit check simultaneously. For strict quotas, guard check-and-increment with a lock. For per-instance quotas, use instance-specific state as in Problem 9.

---
## Problem 18 — Capstone: assemble a callable service with validation, cache, partial, and profiling

**Your task**

Build a small text-processing service that integrates concepts from earlier problems. Requirements: (1) expose a callable `TextService`, (2) accept a raw phrase and normalize it, (3) cache normalization with a bounded cache, (4) use `functools.partial` to create a configurable output formatter, (5) profile service calls using the exception-safe profiler, and (6) verify the results of repeated, normalized-equivalent, and invalid requests. Distinguish service-call counts from cache hits/misses. Use a deterministic fake clock for profiling.

**Before revealing the solution:** predict the edge-case behavior, implement your answer, and write at least one failing test.

### Worked solution and design reasoning

Compose small callables rather than implementing everything inside a monolithic `__call__`. The pipeline normalizes **before** the memoizer, so differently formatted input strings can share one cached key. A `partial` stores the formatter's prefix; the service calls it with a keyword for the normalized value. `SafeProfiler` wraps the whole service, so its call counts do not equal the memoizer's hit count. Invalid non-string input is rejected before cache lookup and is counted as a failed service call.

In [36]:
def normalize_phrase(text):
    if not isinstance(text, str):
        raise TypeError('phrase must be a string')
    return ' '.join(text.strip().casefold().split())


normalization_log = []

def costly_tokenize(normalized):
    normalization_log.append(normalized)
    return tuple(normalized.split())


def format_tokens(*, prefix, tokens):
    return f"{prefix}:" + '|'.join(tokens)


class TextService:
    def __init__(self, *, prefix='TOKENS', cache_size=4):
        self.normalize = Pipeline(normalize_phrase)
        self.tokenize = BoundedMemoizer(costly_tokenize, maxsize=cache_size)
        self.formatter = partial(format_tokens, prefix=prefix)

    def __call__(self, phrase):
        normalized = self.normalize(phrase)
        tokens = self.tokenize(normalized)
        return self.formatter(tokens=tokens)


service = TextService(prefix='INDEX', cache_size=2)
service_clock = iter([10.0, 10.1, 11.0, 11.2, 12.0, 12.3, 13.0, 13.4])
observed_service = SafeProfiler(service, clock=lambda: next(service_clock))

### Executable examples and correctness checks

In [37]:
assert observed_service('  Hello   WORLD ') == 'INDEX:hello|world'
assert observed_service('hello world') == 'INDEX:hello|world'
assert observed_service('Another phrase') == 'INDEX:another|phrase'
assert_raises(TypeError, lambda: observed_service(42), contains='string')
assert normalization_log == ['hello world', 'another phrase']
assert (service.tokenize.hits, service.tokenize.misses) == (1, 2)
assert (observed_service.calls, observed_service.successes, observed_service.failures) == (4, 3, 1)
assert math.isclose(observed_service.total_seconds, 1.0)
assert math.isclose(observed_service.average_seconds, 0.25)
assert observed_service.__wrapped__ is service
print('P18 PASS — compositional callable system, cache behavior and profiling')

P18 PASS — compositional callable system, cache behavior and profiling


**Review / further challenge.** Extend the service with configurable stop-word filtering or a key-aware loader. State a policy for exceptions: failed normalization never reaches the cache, while a failed cache calculation increments misses but is not stored. How would you handle mutable configurations after entries have already been cached?

---
## Additional challenge set (attempt independently)

These are **open-ended extension exercises**. Their solutions are sketched here so that the notebook remains a complete learning resource without pretending that design questions have one universally correct implementation.

| Challenge | Specification | Solution direction / evaluation criterion |
|---|---|---|
| A. Mutable defaults | Return a fresh list for each missing key. | In a `defaultdict`, use a factory that constructs `[]` on every call; test nonidentity of `d['a']` and `d['b']`. |
| B. Profiler reset | Add `reset()` to the profiler. | Reset counts and elapsed time together; test zero-call average and post-reset measurement. |
| C. Cache invalidation | Add `invalidate(*args, **kwargs)` to `BoundedMemoizer`. | Reuse the **same canonical signature-bound key** as `__call__` and delete without executing `fn`. |
| D. Robustness | Make `Retry` use injected backoff. | Inject `sleep_fn` so tests capture requested waits without actually sleeping. |
| E. Async per-instance method | Add both descriptor binding and async profiling. | `__get__` binds the instance; an `async __call__` awaits the underlying method. |
| F. Single-flight recursion | Detect a recursive same-key call. | Track lead-thread identity per in-flight key and raise rather than deadlock; document cross-thread recursion separately. |
| G. Metrics snapshots | Capture a consistent snapshot of thread-safe metrics. | Acquire the lock once to read related counters and totals atomically. |
| H. Production decision | Choose between custom memoizers and `functools.lru_cache`. | Compare key semantics, eviction, thread behavior, introspection, and maintenance costs rather than reinventing features automatically. |

### Design checklist

- Define whether state is **per call**, **per callable instance**, **per decorated function**, or **per method receiver**.
- Treat exceptions as part of the contract: count them, propagate them, and decide whether failed results are cached.
- Avoid global mutable counters for logically independent objects.
- Check signature and keyword override semantics; preserve useful metadata when wrapping.
- Understand lookup rules for special methods and descriptors, and test module functions and instance methods separately.
- Document cache key requirements, eviction, boundedness, and thread-safety constraints.
- Use fake clocks and explicit synchronization instead of brittle performance-threshold tests.
- Prefer standard-library implementations in production when they fully satisfy requirements.

### Final self-check

If every `P01 PASS` through `P18 PASS` appears after **Restart & Run All**, the deterministic assertions for this notebook have succeeded. Tests demonstrate the stated cases; they do not prove comprehensive correctness under all workloads or Python runtimes.